# NB_Silver_Order_Reviews_MongoDB — Bronze to Silver Transformation

## Section 1 — Purpose, Business Context & Verified Profile

- **Source System:** Local MongoDB `retail_reviews.order_reviews` (`99,224` nested BSON documents) → copied via `Copy_MongoDB_Reviews_To_Bronze` to ADLS Gen2 / OneLake Shortcut (`Files/bronze/reviews/order_reviews/` in JSON Lines format).
- **Silver Target:** `silver.order_reviews` (Managed Delta Table) + `silver._quarantine_order_reviews` (Conditional Quarantine).
- **Grain:** Composite Primary Key **`(review_id, order_id)`** (`99,224` unique pairs).
  - **Why Composite PK is Mandatory:** `789` `review_id`s repeat across multiple `order_id`s (`1,603` rows), and `547` `order_id`s have more than one `review_id` (`1,098` rows). Never deduplicate by `review_id` alone or `order_id` alone!
- **Verified Profile (`99,224` documents):**
  - `0` duplicate `(review_id, order_id)`, `0` invalid 32-char hex keys, `0` invalid scores (`1..5`, avg `4.09`), `0` unparseable timestamps, `0` `answer_timestamp < creation_date`.
  - **Comments:** `42,687` reviews (`43.02%`) include a title (`11,566`) or message (`40,950`); remaining `NULL`s are legitimate star-only ratings.
  - **Cross-Table Integrity vs `silver.orders`:** **`0` orphan `order_id`s** (`768` orders out of `99,441` have no review).

> **WARNING — SAFETY LOCK ENABLED (`WRITE_ENABLED = False`):** Run Dry Validation first and inspect counts before setting `WRITE_ENABLED = True`.

## Section 2 — Configuration & JSON Snapshot Auto-Discovery

Automatically discovers the latest `.json` / `.jsonl` snapshot inside `Files/bronze/reviews/order_reviews/`, extracts `INGESTION_RUN_ID` from the filename, and locks writes behind `WRITE_ENABLED = False`.

In [ ]:
# Section 2: Configuration & Snapshot Auto-Discovery
from pyspark.sql import functions as F

WRITE_ENABLED = False      # SAFETY LOCK: flip to True only after Dry Run review
RUN_ORDER_FK_CHECK = True  # Cross-check order_id against silver.orders when present

BRONZE_FOLDER = "Files/bronze/reviews/order_reviews"
json_files = sorted([
    f.name for f in notebookutils.fs.ls(BRONZE_FOLDER)
    if f.name.endswith(".json") or f.name.endswith(".jsonl")
])
if not json_files:
    raise FileNotFoundError(f"No JSON/JSONL snapshots found in {BRONZE_FOLDER}")

LATEST_FILE = json_files[-1]
BRONZE_PATH = f"{BRONZE_FOLDER}/{LATEST_FILE}"
INGESTION_RUN_ID = LATEST_FILE.rsplit("_", 1)[-1].split(".")[0] if "_" in LATEST_FILE else "manual_run"

SILVER_SCHEMA = "silver"
SILVER_TARGET_TABLE = f"{SILVER_SCHEMA}.order_reviews"
QUARANTINE_TARGET_TABLE = f"{SILVER_SCHEMA}._quarantine_order_reviews"
SILVER_ORDERS_TABLE = f"{SILVER_SCHEMA}.orders"

SOURCE_SYSTEM = "mongodb_retail_reviews"
SOURCE_TABLE = "retail_reviews.order_reviews"

print(f"Discovered JSON snapshots  : {len(json_files)}")
print(f"Selected BRONZE_PATH       : {BRONZE_PATH}")
print(f"Extracted INGESTION_RUN_ID : {INGESTION_RUN_ID}")
print(f"Configured SILVER_TARGET   : {SILVER_TARGET_TABLE}")
print(f"WRITE_ENABLED              : {WRITE_ENABLED}")

## Section 3 — Distributed Bronze JSON Ingestion (Nested BSON Schema)

Reads the JSON Lines file directly with PySpark and inspects the nested MongoDB schema (`comment.*`, `timestamps.*`, `source_lineage.*`).

In [ ]:
# Section 3: Read Bronze JSON & Inspect Nested Schema
df_bronze_raw = (
    spark.read
    .format("json")
    .load(BRONZE_PATH)
    .withColumn("_source_file", F.input_file_name())
)

bronze_count = df_bronze_raw.count()
print(f"Bronze MongoDB Review Count: {bronze_count:,} (Expected: 99,224)")
df_bronze_raw.printSchema()
display(df_bronze_raw.limit(3))

## Stage 1 & 2 — Flatten Nested BSON Structs, Normalize & Parse Timestamps

Safely flattens `comment.*`, `timestamps.*`, and `source_lineage.*` into typed columns:
1. Lowercases and trims `review_id` and `order_id`.
2. Safe-casts `review_score` (`INT`), `review_creation_date` (`TIMESTAMP`), and `review_answer_timestamp` (`TIMESTAMP`).
3. Maps whitespace-only `comment.title` / `comment.message` to `NULL` while preserving UTF-8 Portuguese text verbatim.
4. Computes analytical columns `has_comment_text`, `is_negative_review` (`review_score <= 2`), and `review_response_hours`.

In [ ]:
# Stage 1 & 2: Flatten Nested BSON Structs & Standardize
required_top_cols = ["_id", "review_id", "order_id", "review_score", "comment", "timestamps"]
missing_top = [c for c in required_top_cols if c not in df_bronze_raw.columns]
if missing_top:
    raise ValueError(f"Stage 1 FAILED — Missing top-level MongoDB fields: {missing_top}")

def clean_comment_col(col_expr):
    s = col_expr.cast("string")
    return F.when(s.isNull() | (F.length(F.trim(s)) == 0), F.lit(None).cast("string")).otherwise(s)

df_std = (
    df_bronze_raw
    .select(
        F.col("_id").cast("string").alias("review_document_id"),
        F.lower(F.trim(F.col("review_id").cast("string"))).alias("review_id"),
        F.lower(F.trim(F.col("order_id").cast("string"))).alias("order_id"),
        F.expr("try_cast(review_score AS INT)").alias("review_score"),
        clean_comment_col(F.col("comment.title")).alias("review_comment_title"),
        clean_comment_col(F.col("comment.message")).alias("review_comment_message"),
        F.to_timestamp(F.col("timestamps.review_creation_date_raw").cast("string"), "yyyy-MM-dd HH:mm:ss").alias("review_creation_date"),
        F.to_timestamp(F.col("timestamps.review_answer_timestamp_raw").cast("string"), "yyyy-MM-dd HH:mm:ss").alias("review_answer_timestamp"),
        F.col("source_lineage.source_file").cast("string").alias("mongo_source_file"),
        F.col("source_lineage.ingestion_run_id").cast("string").alias("mongo_ingestion_run_id"),
        F.col("_source_file"),
    )
    .withColumn("has_comment_text", F.col("review_comment_title").isNotNull() | F.col("review_comment_message").isNotNull())
    .withColumn("is_negative_review", F.col("review_score").isin(1, 2))
    .withColumn(
        "review_response_hours",
        F.round((F.unix_timestamp("review_answer_timestamp") - F.unix_timestamp("review_creation_date")) / 3600.0, 2)
    )
)

print("Stage 1 & 2 Flattening and Standardization: PASSED")
display(df_std.groupBy("review_score").count().orderBy("review_score"))

## Stage 3 & 4 — Composite PK `(review_id, order_id)`, `silver.orders` FK Check & Quarantine

1. Validates `^[0-9a-f]{32}$` on both `review_id` and `order_id`, `review_score BETWEEN 1 AND 5`, and non-null timestamps.
2. Enforces composite PK `(review_id, order_id)` uniqueness (`0` duplicates).
3. Cross-checks `order_id` against `silver.orders` when present (`0` orphans expected) and routes any invalid record to `df_quarantine`.

In [ ]:
# Stage 3 & 4: Composite PK, Orders FK Check & Quarantine Routing
hex32_regex = r"^[0-9a-f]{32}$"

dup_pk_df = (
    df_std.filter(F.col("review_id").isNotNull() & F.col("order_id").isNotNull())
    .groupBy("review_id", "order_id").count().filter(F.col("count") > 1)
    .select("review_id", "order_id").withColumn("_dup_pk", F.lit(True))
)
dup_pk_cnt = dup_pk_df.count()

if RUN_ORDER_FK_CHECK and spark.catalog.tableExists(SILVER_ORDERS_TABLE):
    orders_ref = spark.table(SILVER_ORDERS_TABLE).select("order_id").distinct()
    missing_ord_df = (
        df_std.select("order_id").distinct()
        .join(orders_ref, on="order_id", how="left_anti")
        .withColumn("_missing_order_fk", F.lit(True))
    )
    print(f"Referential Check vs {SILVER_ORDERS_TABLE}: {missing_ord_df.count()} orphan order_id(s) (Expected: 0)")
else:
    missing_ord_df = spark.createDataFrame([], "order_id string, _missing_order_fk boolean")
    print(f"Referential Check vs {SILVER_ORDERS_TABLE} skipped (table not found or disabled).")

df_checked = (
    df_std
    .join(dup_pk_df, on=["review_id", "order_id"], how="left")
    .withColumn("_dup_pk", F.coalesce(F.col("_dup_pk"), F.lit(False)))
    .join(missing_ord_df, on="order_id", how="left")
    .withColumn("_missing_order_fk", F.coalesce(F.col("_missing_order_fk"), F.lit(False)))
    .withColumn(
        "dq_rejection_reason",
        F.concat_ws(
            " | ",
            F.when(~F.coalesce(F.col("review_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_REVIEW_ID_HEX32")),
            F.when(~F.coalesce(F.col("order_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_ORDER_ID_HEX32")),
            F.when(F.col("review_score").isNull() | (~F.col("review_score").between(1, 5)), F.lit("INVALID_REVIEW_SCORE")),
            F.when(F.col("review_creation_date").isNull(), F.lit("NULL_OR_INVALID_CREATION_DATE")),
            F.when(F.col("review_answer_timestamp").isNull(), F.lit("NULL_OR_INVALID_ANSWER_TS")),
            F.when(F.col("_dup_pk"), F.lit("DUPLICATE_COMPOSITE_PK")),
            F.when(F.col("_missing_order_fk"), F.lit("MISSING_ORDER_FK")),
        )
    )
)

df_valid = df_checked.filter(F.length(F.col("dq_rejection_reason")) == 0)
df_quarantine = (
    df_checked.filter(F.length(F.col("dq_rejection_reason")) > 0)
    .select(
        "review_document_id", "review_id", "order_id", "review_score",
        "review_comment_title", "review_comment_message",
        "review_creation_date", "review_answer_timestamp",
        "dq_rejection_reason", "_source_file"
    )
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("quarantined_at", F.current_timestamp())
)

valid_count = df_valid.count()
quarantine_count = df_quarantine.count()
print(f"Duplicate (review_id, order_id) : {dup_pk_cnt} (Expected: 0)")
print(f"Valid Review Documents          : {valid_count:,} (Expected: 99,224)")
print(f"Quarantined Rows                : {quarantine_count:,} (Expected: 0)")

## Stage 5 — Full Lineage, Diagnostics & Reconciliation

Projects the `17`-column Silver contract (`review_document_id`, `review_id`, `order_id`, `review_score`, comments, timestamps, `3` analytical columns, `2` MongoDB origin columns, and `4` standardized Silver lineage columns) and asserts `bronze_count == silver_count + quarantine_count`.

In [ ]:
# Stage 5: Lineage, Diagnostics & Reconciliation
df_silver = (
    df_valid
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("silver_processed_at", F.current_timestamp())
    .select(
        "review_document_id",
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp",
        "has_comment_text",
        "is_negative_review",
        "review_response_hours",
        "mongo_source_file",
        "mongo_ingestion_run_id",
        "source_system",
        "source_table",
        "ingestion_run_id",
        "silver_processed_at",
    )
)

silver_count = df_silver.count()
assert bronze_count == silver_count + quarantine_count, (
    f"Reconciliation mismatch: bronze={bronze_count}, silver={silver_count}, quarantine={quarantine_count}"
)

diag = df_silver.agg(
    F.countDistinct("review_id").alias("distinct_reviews"),
    F.countDistinct("order_id").alias("distinct_orders"),
    F.sum(F.when(F.col("has_comment_text"), 1).otherwise(0)).alias("with_comment"),
    F.sum(F.when(F.col("is_negative_review"), 1).otherwise(0)).alias("negative_reviews"),
    F.round(F.avg("review_score"), 4).alias("avg_score"),
).collect()[0]

print(f"Stage 5 Reconciliation PASSED : bronze={bronze_count:,}, silver={silver_count:,}, quarantine={quarantine_count:,}")
print(f"Distinct review_id / order_id : {diag['distinct_reviews']:,} / {diag['distinct_orders']:,} (Expected: 98,410 / 98,673)")
print(f"Reviews with Comment Text     : {diag['with_comment']:,} (Expected: 42,687)")
print(f"Negative Reviews (1-2 stars)  : {diag['negative_reviews']:,} (Expected: 14,575)")
print(f"Average Review Score          : {diag['avg_score']} (Expected: 4.0864)")
df_silver.printSchema()

## Section 5 — Guarded Delta Write (`silver.order_reviews`)

Creates `silver` schema if missing and persists `silver.order_reviews` strictly when `WRITE_ENABLED = True`.

In [ ]:
# Section 5: Guarded Delta Write
if WRITE_ENABLED:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TARGET_TABLE)
    print(f"Wrote {silver_count:,} rows to {SILVER_TARGET_TABLE}")

    if quarantine_count > 0:
        df_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(QUARANTINE_TARGET_TABLE)
        print(f"Wrote {quarantine_count:,} rows to {QUARANTINE_TARGET_TABLE}")
    else:
        print("Zero quarantined rows — skipping quarantine table write.")
else:
    print("WRITE_ENABLED is False — dry validation only; no Silver or Quarantine data was written.")

## Section 6 — Post-Write Delta Read-Back Verification

Reads back `silver.order_reviews` when `WRITE_ENABLED = True` and confirms `99,224` rows and `0` duplicate `(review_id, order_id)` keys.

In [ ]:
# Section 6: Post-Write Validation
if WRITE_ENABLED:
    df_check = spark.table(SILVER_TARGET_TABLE)
    chk_count = df_check.count()
    chk_dup = df_check.groupBy("review_id", "order_id").count().filter(F.col("count") > 1).count()
    assert chk_count == silver_count, f"Row mismatch: {chk_count} vs {silver_count}"
    assert chk_dup == 0, f"Duplicate composite PK found: {chk_dup}"
    print(f"Post-write validation PASSED — rows={chk_count:,}, dup_pk={chk_dup}")
    display(df_check.limit(5))
else:
    print("Post-write validation skipped because WRITE_ENABLED is False.")